# Revisão: KNN — Normalização e Recomendação
Continuação da revisão de KNN.

Esta revisão cobre:
- Por que normalizar é obrigatório no KNN
- Pipeline completo: scaler + modelo + save + load
- KNN para recomendação por similaridade

In [ ]:
import math
import pandas as pd
import joblib
from pathlib import Path
from sklearn.datasets import load_wine
from sklearn.neighbors import KNeighborsClassifier, NearestNeighbors
from sklearn.preprocessing import StandardScaler

## Por que Normalizar é Obrigatório no KNN
O KNN decide por distância. Se uma feature tem escala muito maior que outra,
ela domina o cálculo e as demais perdem importância.

**Exemplo do problema:**
Dois clientes quase idênticos em salário (R$ 5.000 vs R$ 5.001),
mas com histórico de pagamento completamente oposto (9 vs 1).

Sem normalização, o KNN os trata como vizinhos próximos — o que está errado.

In [ ]:
# Dois clientes: escalas completamente diferentes
# salary em R$ (milhares) vs historico de 0 a 10
p1 = {"salary": 5000, "historico": 9}  # bom pagador
p2 = {"salary": 5001, "historico": 1}  # pessimo pagador

dx = p1["salary"] - p2["salary"]
dy = p1["historico"] - p2["historico"]
dist = math.sqrt(dx**2 + dy**2)

print(f"Distancia sem normalizar: {dist:.2f}")
print()
print("O KNN calcula distancia ~1.0, como se fossem quase identicos.")
print("Mas historico 9 vs 1 sao clientes COMPLETAMENTE diferentes!")
print("O salario (grande em escala) mascarou o historico (pequeno em escala).")

## KNN com Normalização (StandardScaler)
`StandardScaler` coloca todas as features na mesma escala:
média zero e desvio padrão igual a 1.

Assim nenhuma feature domina o cálculo de distância.

In [ ]:
# Dataset com escala mista: salary em R$ e historico de 0 a 10
df = pd.DataFrame([
    {"salary": 4000, "historico": 8, "aprovado": 1},
    {"salary": 1000, "historico": 2, "aprovado": 0},
    {"salary": 6000, "historico": 9, "aprovado": 1},
    {"salary": 2000, "historico": 3, "aprovado": 0},
    {"salary": 5000, "historico": 7, "aprovado": 1},
    {"salary": 1500, "historico": 1, "aprovado": 0},
    {"salary": 7000, "historico": 8, "aprovado": 1},
    {"salary": 3000, "historico": 4, "aprovado": 0},
])

X = df[["salary", "historico"]]
y = df["aprovado"]

# fit_transform: aprende a escala e transforma em uma unica chamada
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Treina o KNN nos dados normalizados
modelo = KNeighborsClassifier(n_neighbors=3)
modelo.fit(X_scaled, y)

print("Modelo treinado com dados normalizados.")
print()
print("Media de cada feature (aprox. 0 apos normalizar):")
print(X_scaled.mean(axis=0).round(2))

In [ ]:
# Testando os dois clientes do exemplo anterior
novos = pd.DataFrame([
    {"salary": 5000, "historico": 9},  # bom pagador
    {"salary": 5001, "historico": 1},  # pessimo pagador
])

# transform (sem fit): aplica a mesma escala aprendida no treino
novos_scaled = scaler.transform(novos)
previsoes = modelo.predict(novos_scaled)

for i, prev in enumerate(previsoes):
    label = "Aprovado" if prev == 1 else "Reprovado"
    print(f"Cliente {i+1}: {label}")

## Salvando e Carregando o Modelo
Igual ao fluxo do MLP: salvamos o modelo e o scaler juntos.
Na hora de usar em produção, só carregamos — sem re-treinar.

In [ ]:
# Salva modelo e scaler juntos
Path("models").mkdir(exist_ok=True)
joblib.dump({"modelo": modelo, "scaler": scaler}, "models/revisao_knn.joblib")
print("Salvo em models/revisao_knn.joblib")

In [ ]:
# Carrega em outro contexto (simula producao)
state = joblib.load("models/revisao_knn.joblib")
modelo_prod = state["modelo"]
scaler_prod = state["scaler"]

# Novo cliente chegando
novo = pd.DataFrame([{"salary": 6500, "historico": 8}])
novo_scaled = scaler_prod.transform(novo)
prev = modelo_prod.predict(novo_scaled)[0]

print(f"Previsao em producao: {'Aprovado' if prev == 1 else 'Reprovado'}")

## KNN para Recomendação por Similaridade (Cenário de Produção)
No modo de **recomendação**, não precisamos de rótulos (aprovado/reprovado).
Queremos saber: quais itens são mais parecidos com este?

Usamos `NearestNeighbors` para isso.

**Exemplo prático:**
em um catálogo de bebidas, dado um vinho que o cliente gostou,
recomendar vinhos com perfil químico parecido.

Para evitar dataset manual, vamos usar o `load_wine` do `sklearn`.

In [ ]:
# Dataset pronto do sklearn: caracteristicas quimicas de vinhos
wine = load_wine()

# Monta DataFrame com as features
vinhos = pd.DataFrame(wine.data, columns=wine.feature_names)

# Adiciona informacoes uteis para exibicao
vinhos["tipo"] = [wine.target_names[t] for t in wine.target]
vinhos["id_vinho"] = vinhos.index

print("Amostra do catalogo de vinhos:")
print(vinhos[["id_vinho", "tipo", "alcohol", "malic_acid", "color_intensity"]].head(8))

In [ ]:
# Features numericas para calcular similaridade
X_rec = vinhos[wine.feature_names]

# Normaliza para nenhuma feature dominar a distancia
scaler_rec = StandardScaler()
X_rec_scaled = scaler_rec.fit_transform(X_rec)

# Parametro facil de testar
metrica_rec = "euclidean"  # troque para "manhattan" e compare

# NearestNeighbors: busca os mais proximos, sem precisar de rotulos
modelo_rec = NearestNeighbors(n_neighbors=4, metric=metrica_rec)
modelo_rec.fit(X_rec_scaled)

# Cliente gostou do vinho id 10
id_referencia = 10
referencia = X_rec_scaled[[id_referencia]]
distancias, indices = modelo_rec.kneighbors(referencia)

print(f"Metrica usada: {metrica_rec}")
print(f"Vinho de referencia (id={id_referencia}, tipo={vinhos.iloc[id_referencia]['tipo']})")
print()
print("Vinhos mais parecidos (incluindo ele mesmo na posicao 0):")
for i, idx in enumerate(indices[0]):
    tipo = vinhos.iloc[idx]["tipo"]
    dist = distancias[0][i]
    print(f"  id={idx} | tipo={tipo} | distancia={dist:.3f}")

In [ ]:
# Funcao reutilizavel para recomendacao por id do catalogo
def recomendar_vinhos(id_vinho, n=3):
    referencia = X_rec_scaled[[id_vinho]]

    # kneighbors retorna distancias e indices dos vizinhos
    distancias, indices = modelo_rec.kneighbors(referencia, n_neighbors=n + 1)

    tipo_ref = vinhos.iloc[id_vinho]["tipo"]
    print(f"Quem gostou do vinho id={id_vinho} (tipo={tipo_ref}) pode gostar de:")
    for i, idx in enumerate(indices[0]):
        if idx == id_vinho:
            continue  # pula o proprio vinho de referencia
        tipo = vinhos.iloc[idx]["tipo"]
        dist = distancias[0][i]
        print(f"  - id={idx} | tipo={tipo} | distancia={dist:.3f}")

recomendar_vinhos(10)
print()
recomendar_vinhos(130)

---
## Resumo Final

| Situação | O que usar |
|---|---|
| Features com escalas diferentes | `StandardScaler` antes do KNN |
| Classificação (aprovado/reprovado) | `KNeighborsClassifier` |
| Encontrar itens similares | `NearestNeighbors` |
| Exemplo pronto para praticar | `load_wine` do `sklearn` |
| Reutilizar sem re-treinar | `joblib.dump` / `joblib.load` |

O fluxo de produção é sempre:
**treino -> salva -> carrega -> transforma -> prediz**